# model_experiment_LightGBM

Global gradient-boosting model over all 3331 series. Direct multi-horizon: every lag feature is >= 39 weeks back, so one `predict()` covers the whole 39-week test set with no recursion.

MLflow experiment: **LightGBM_Training** — runs: Cleaning, CV_*, Feature_Importance, Final.

In [1]:
!pip install -q mlflow
!pip install -q mlflow lightgbm scikit-learn

import os
if not os.path.exists("walmart-sales-forecasting"):
    !git clone https://github.com/ekatsirekidze/walmart-sales-forecasting.git
import sys; sys.path.insert(0, "/kaggle/working/walmart-sales-forecasting")

import glob, shutil, zipfile
src = glob.glob("/kaggle/input/*walmart*") + glob.glob("/kaggle/input/*/*walmart*")
assert src, "competition data not attached"
os.makedirs("data", exist_ok=True)
for f in glob.glob(src[0] + "/*"):
    (zipfile.ZipFile(f).extractall("data") if f.endswith(".zip") else shutil.copy(f, "data"))
print("data/:", os.listdir("data"))

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 754.0 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 53.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 81.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 62.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.3/99.3 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.6/954.6 kB 41.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.9/214.9 kB 12.2 MB/s eta 0:00:00
Cloning into 'walmart-sales-forecasting'...
r

In [2]:
import os
from kaggle_secrets import UserSecretsClient

s = UserSecretsClient()
os.environ["MLFLOW_TRACKING_URI"] = s.get_secret("MLFLOW_TRACKING_URI")
os.environ["MLFLOW_TRACKING_USERNAME"] = s.get_secret("MLFLOW_TRACKING_USERNAME")
os.environ["MLFLOW_TRACKING_PASSWORD"] = s.get_secret("MLFLOW_TRACKING_PASSWORD")

In [3]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path().resolve()))  # repo root on path

import numpy as np
import pandas as pd
import mlflow

from src.data import load_raw, make_submission
from src.metrics import wmae_report
from src.validation import FOLDS, split_fold
from src.mlflow_utils import setup_mlflow

train, test, features, stores = load_raw("data")
print(train.shape, test.shape)

from lightgbm import LGBMRegressor
from sklearn.pipeline import Pipeline

from src.preprocessing import WalmartPreprocessor, make_xyw
from src.postprocess import apply_christmas_shift
from src.mlflow_utils import REGISTRY_MODEL_NAME

setup_mlflow("LightGBM_Training")

(421570, 5) (115064, 4)
MLflow -> https://dagshub.com/ekatsirekidze/walmart-sales-forecasting.mlflow | experiment: LightGBM_Training


## Run 1 — Cleaning (documenting the data decisions)

In [4]:
with mlflow.start_run(run_name="LightGBM_Cleaning"):
    mlflow.log_params({
        "negative_sales": "kept (returns are real; metric is L1 on raw dollars)",
        "markdown_na": "fill 0 + <col>_present flag (NA = no promo running)",
        "cpi_unemployment_na": "forward-fill per store (covers test tail)",
        "target_transform": "none (L1 objective on raw sales matches WMAE)",
        "n_rows": len(train),
        "n_series": train.groupby(["Store", "Dept"]).ngroups,
    })

🏃 View run LightGBM_Cleaning at: https://dagshub.com/ekatsirekidze/walmart-sales-forecasting.mlflow/#/experiments/1/runs/a5865d8ce1f3400c8e54a116744a341f
🧪 View experiment at: https://dagshub.com/ekatsirekidze/walmart-sales-forecasting.mlflow/#/experiments/1


## Run 2 — Cross-validation on the shared folds

`sample_weight = 1 + 4*IsHoliday` so training optimizes what Kaggle measures.

In [5]:
def eval_fold(params, fold):
    tr, va = split_fold(train, fold)
    Xtr, ytr, wtr = make_xyw(tr)
    Xva, yva, _ = make_xyw(va)
    prep = WalmartPreprocessor(features_df=features, stores_df=stores)
    Xtr_t = prep.fit(Xtr, ytr).transform(Xtr)
    Xva_t = prep.transform(Xva)
    model = LGBMRegressor(**params)
    model.fit(Xtr_t, ytr, sample_weight=wtr)
    rep = wmae_report(yva, model.predict(Xva_t), va["IsHoliday"])
    return rep, model, Xtr_t

In [6]:
PARAM_GRID = [
    dict(objective="l1", n_estimators=800, learning_rate=0.05, num_leaves=127,
         min_child_samples=30, colsample_bytree=0.8, random_state=42, verbose=-1),
    dict(objective="l1", n_estimators=1500, learning_rate=0.03, num_leaves=255,
         min_child_samples=20, colsample_bytree=0.8, random_state=42, verbose=-1),
    dict(objective="l1", n_estimators=800, learning_rate=0.05, num_leaves=63,
         min_child_samples=50, colsample_bytree=0.9, random_state=42, verbose=-1),
]

results = []
for i, params in enumerate(PARAM_GRID):
    with mlflow.start_run(run_name=f"LightGBM_CV_{i}"):
        mlflow.log_params({**params, "fold": 1})
        rep, model, _ = eval_fold(params, fold=1)
        mlflow.log_metrics(rep)
    results.append((i, rep["wmae"]))
    print(i, rep)
best_i = min(results, key=lambda r: r[1])[0]
print("best config:", best_i)

🏃 View run LightGBM_CV_0 at: https://dagshub.com/ekatsirekidze/walmart-sales-forecasting.mlflow/#/experiments/1/runs/1019cc9311694f7ca8a1f00605a0048f
🧪 View experiment at: https://dagshub.com/ekatsirekidze/walmart-sales-forecasting.mlflow/#/experiments/1
0 {'wmae': 1924.7996309158748, 'mae_holiday': 2355.1343582740337, 'mae_nonholiday': 1743.0279113584716}
🏃 View run LightGBM_CV_1 at: https://dagshub.com/ekatsirekidze/walmart-sales-forecasting.mlflow/#/experiments/1/runs/270c35fda3a54e728a038e2e0991d9e3
🧪 View experiment at: https://dagshub.com/ekatsirekidze/walmart-sales-forecasting.mlflow/#/experiments/1
1 {'wmae': 1978.763122690417, 'mae_holiday': 2540.4030879955367, 'mae_nonholiday': 1741.5285799603712}
🏃 View run LightGBM_CV_2 at: https://dagshub.com/ekatsirekidze/walmart-sales-forecasting.mlflow/#/experiments/1/runs/b8c9588a1d244b4a918d6ca1c829f76d
🧪 View experiment at: https://dagshub.com/ekatsirekidze/walmart-sales-forecasting.mlflow/#/experiments/1
2 {'wmae': 1942.982010548117

In [7]:
# sanity-check the winner on Fold 2 (regular weeks, no big holidays)
with mlflow.start_run(run_name="LightGBM_CV_best_fold2"):
    mlflow.log_params({**PARAM_GRID[best_i], "fold": 2})
    rep2, _, _ = eval_fold(PARAM_GRID[best_i], fold=2)
    mlflow.log_metrics(rep2)
print(rep2)

🏃 View run LightGBM_CV_best_fold2 at: https://dagshub.com/ekatsirekidze/walmart-sales-forecasting.mlflow/#/experiments/1/runs/932cc17dfb8f4d6b8b0ad4b50e761daf
🧪 View experiment at: https://dagshub.com/ekatsirekidze/walmart-sales-forecasting.mlflow/#/experiments/1
{'wmae': 1570.545998348542, 'mae_holiday': 1610.5922653255698, 'mae_nonholiday': 1559.646805901295}


## Run 3 — Feature importance

In [8]:
rep, model, Xtr_t = eval_fold(PARAM_GRID[best_i], fold=1)
imp = pd.Series(model.feature_importances_, index=Xtr_t.columns).sort_values(ascending=False)
with mlflow.start_run(run_name="LightGBM_Feature_Importance"):
    mlflow.log_dict(imp.to_dict(), "feature_importance.json")
    mlflow.log_metrics(rep)
print(imp.head(20))

🏃 View run LightGBM_Feature_Importance at: https://dagshub.com/ekatsirekidze/walmart-sales-forecasting.mlflow/#/experiments/1/runs/9a0a8e3ad2a54691ba1ca0bca6213029
🧪 View experiment at: https://dagshub.com/ekatsirekidze/walmart-sales-forecasting.mlflow/#/experiments/1
Dept                 15856
Store                14108
dept_woy_mean         6859
lag_39                6644
Temperature           5997
series_std            5447
lag_46                5146
Fuel_Price            4327
series_mean           4200
series_median         3854
lag_52                3766
lag_53                3319
CPI                   3236
Unemployment          3137
lag_51                2770
yearly_smooth         2254
days_to_christmas     1934
weekofyear            1803
Size                  1253
store_mean            1250
dtype: int32


## Run 4 — Final: fit Pipeline on ALL train, log to MLflow, predict raw test

The pipeline (preprocessing + model in one object) is what gets registered — it runs directly on the un-preprocessed test set.

In [9]:
BEST = PARAM_GRID[best_i]
X, y, w = make_xyw(train)
pipe = Pipeline([
    ("prep", WalmartPreprocessor(features_df=features, stores_df=stores)),
    ("model", LGBMRegressor(**BEST)),
])
pipe.fit(X, y, model__sample_weight=w)

with mlflow.start_run(run_name="LightGBM_Final"):
    mlflow.log_params(BEST)
    mlflow.log_metric("fold1_wmae", min(r[1] for r in results))
    # if LightGBM ends up the overall best architecture, add:
    #   registered_model_name=REGISTRY_MODEL_NAME
    mlflow.sklearn.log_model(pipe, name="model",
                             serialization_format="cloudpickle",
                             registered_model_name="walmart-lightgbm")

2026/07/07 15:44:24 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'walmart-lightgbm' already exists. Creating a new version of this model...
2026/07/07 15:44:47 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: walmart-lightgbm, version 2
Created version '2' of model 'walmart-lightgbm'.


🏃 View run LightGBM_Final at: https://dagshub.com/ekatsirekidze/walmart-sales-forecasting.mlflow/#/experiments/1/runs/41c002c790c143d791d55bbfc068db46
🧪 View experiment at: https://dagshub.com/ekatsirekidze/walmart-sales-forecasting.mlflow/#/experiments/1


In [10]:
# raw test in -> submission out
raw_test = test[["Store", "Dept", "Date"]]
sub = test[["Store", "Dept", "Date"]].copy()
sub["pred"] = pipe.predict(raw_test)
sub = apply_christmas_shift(sub, pred_col="pred")
make_submission(sub, "pred", "submission_lightgbm.csv")
print("wrote submission_lightgbm.csv")

wrote submission_lightgbm.csv
